# Cleft Lip Regeneration Training on Colab

This notebook allows you to continue training the mask-aware inpainting model on Google Colab.

## Setup Instructions

1. Upload your project folder to Google Drive (e.g., to `My Drive/Colab Projects/cleft-lip-regeneration`)
2. Mount your Drive in this notebook
3. Set the project path and run the cells below

## Requirements

- Your project folder should contain:
  - `data/celeba/` with images
  - `data/masks/` with mask PNGs
  - `checkpoints/` with existing checkpoints
  - `src/` with source code
  - `requirements.txt`

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set your project path on Drive
# Change this to your actual path
PROJECT_PATH = '/content/drive/My Drive/Colab Projects/cleft-lip-regeneration'  # Update this path

# Change to project directory
import os
os.chdir(PROJECT_PATH)
print(f"Current directory: {os.getcwd()}")

# List contents to verify
!ls -la

In [ ]:
# Install dependencies
!pip install -r requirements.txt

In [ ]:
# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# Check existing checkpoints
import glob
checkpoints = sorted(glob.glob('checkpoints/epoch_*.pt'))
print(f"Found {len(checkpoints)} checkpoints:")
for ckpt in checkpoints[-5:]:  # Show last 5
    print(ckpt)

In [ ]:
# Training configuration
# Adjust these parameters as needed

EPOCHS = 50  # Total epochs (will resume from latest checkpoint)
BATCH_SIZE = 8  # Adjust based on GPU memory
IMAGE_SIZE = 256
USE_PERCEPTUAL = True  # Use perceptual loss for better quality
LAMBDA_PERCEPTUAL = 0.1
LAMBDA_MASK = 10.0
LOG_EVERY = 25
NUM_WORKERS = 2  # Colab has limited CPU cores

# Auto-resume from latest checkpoint
import glob
checkpoints = sorted(glob.glob('checkpoints/epoch_*.pt'))
if checkpoints:
    RESUME_FROM = checkpoints[-1]
    print(f"Resuming from: {RESUME_FROM}")
else:
    RESUME_FROM = ""
    print("No checkpoints found, starting from scratch")

In [ ]:
# Start training
# This will run for the specified number of epochs
# You can interrupt and restart if needed

import subprocess
import sys

cmd = [
    sys.executable, '-m', 'src.train',
    '--image-root', 'data/celeba',
    '--mask-root', 'data/masks',
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--image-size', str(IMAGE_SIZE),
    '--save-dir', 'checkpoints',
    '--num-workers', str(NUM_WORKERS),
    '--log-every', str(LOG_EVERY),
    '--lambda-perceptual', str(LAMBDA_PERCEPTUAL),
    '--lambda-mask', str(LAMBDA_MASK),
]

if USE_PERCEPTUAL:
    cmd.append('--use-perceptual')

if RESUME_FROM:
    cmd.extend(['--resume', RESUME_FROM])

print("Running command:", ' '.join(cmd))

# Run training
result = subprocess.run(cmd, capture_output=False, text=True)

if result.returncode == 0:
    print("Training completed successfully!")
else:
    print(f"Training failed with return code {result.returncode}")
    print("STDOUT:", result.stdout)
    print("STDERR:", result.stderr)

In [ ]:
# Check training results
import glob
import torch

# Load best model
best_model_path = 'artifacts/best.pt'
if os.path.exists(best_model_path):
    checkpoint = torch.load(best_model_path, map_location='cpu')
    print(f"Best model from epoch: {checkpoint.get('epoch', 'unknown')}")
    metrics = checkpoint.get('metrics', {})
    print(f"Validation loss: {metrics.get('val_loss', 'N/A')}")
    print(f"PSNR: {metrics.get('psnr', 'N/A')}")
    print(f"SSIM: {metrics.get('ssim', 'N/A')}")
else:
    print("Best model not found")

# List all checkpoints
checkpoints = sorted(glob.glob('checkpoints/epoch_*.pt'))
print(f"\nTotal checkpoints: {len(checkpoints)}")
print("Latest checkpoints:")
for ckpt in checkpoints[-5:]:
    print(ckpt)

In [ ]:
# Optional: Test inference on a sample
# Make sure you have test images in the data folder

import glob
from PIL import Image
import matplotlib.pyplot as plt

# Find a sample image and mask
sample_images = glob.glob('data/celeba/*.jpg')[:1]
sample_masks = glob.glob('data/masks/*.png')[:1]

if sample_images and sample_masks:
    sample_image = sample_images[0]
    sample_mask = sample_masks[0]
    
    print(f"Testing with: {sample_image} and {sample_mask}")
    
    # Run inference
    output_path = 'test_output.png'
    cmd = [
        sys.executable, '-m', 'src.infer',
        '--weights', 'artifacts/best.pt',
        '--image', sample_image,
        '--mask', sample_mask,
        '--output', output_path,
        '--image-size', str(IMAGE_SIZE)
    ]
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        print(f"Inference successful! Output saved to {output_path}")
        
        # Display result
        if os.path.exists(output_path):
            img = Image.open(output_path)
            plt.imshow(img)
            plt.title('Generated Image')
            plt.show()
    else:
        print("Inference failed:")
        print(result.stderr)
else:
    print("No sample images/masks found for testing")

## Downloading Results

After training, you can download the updated checkpoints and best model from your Google Drive.

The checkpoints will be saved in `checkpoints/` and the best model in `artifacts/best.pt`.

## Tips for Colab Training

- Colab sessions timeout after ~12 hours, so save frequently
- Use smaller batch sizes if you run out of GPU memory
- Monitor the training output for progress
- You can restart training from any checkpoint by updating the RESUME_FROM variable